# 예제 03. 흩어진 코드를 패키지로 — `kmu_llm.py` · Example 03 — Turning Scattered Code into a Package

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 40분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 40 minutes

## 목표 · Objectives
- 1주차에 복사해 붙이던 코드를 **파일 하나**로 정리한다 (리팩터링)
- 자신의 저장소에 올린다
- **런타임을 초기화한 뒤** 내려받아 `import` 가 되는지 확인한다
- 모듈 캐시 함정(`importlib.reload`)을 안다

> **EN**
> - Refactor the code you copy-pasted all through week 1 into **a single file**
> - Push it to your own repository
> - **Reset the runtime**, download it again, and confirm the `import` works
> - Learn the module-cache trap (`importlib.reload`)

> 오늘의 핵심 실습입니다. 3주차부터 이 파일을 계속 키워 나갑니다.
> **EN** — This is today's central lab. From week 3 on we keep growing this file.

---
## 1. 무엇을 모을 것인가 · What are we collecting?

1주차 노트북 네 개에 흩어져 있던 것들입니다.

| 지난주 코드 | 하는 일 | 정리 후 |
|---|---|---|
| `get_secret()` | Secrets/환경변수/입력에서 값 찾기 | 그대로 함수 |
| `get_base_url()` | 서버 주소 정리 | 그대로 함수 |
| `MODEL` 상수 | 서버 설정 | `PROVIDERS` 딕셔너리로 통합 |
| `client = OpenAI(...)` | 클라이언트 생성 | `LLM` 클래스의 `__init__` |
| `ask()` | 한 줄 질문 | `LLM.ask()` 메서드 |
| `ask_retry()` | 실패 재시도 | `LLM.ask_retry()` 메서드 |
| 서버 진단 코드 | 모델 목록 확인 | `check_server()` 함수 |

**EN** — What was scattered across last week's four notebooks:

| Last week's code | What it does | After the refactor |
|---|---|---|
| `get_secret()` | Find a value in Secrets / env / input | Stays a function |
| `get_base_url()` | Tidy the server URL | Stays a function |
| The `MODEL` constant | Server configuration | Folded into the `PROVIDERS` dict |
| `client = OpenAI(...)` | Create the client | `LLM.__init__` |
| `ask()` | One-line question | The `LLM.ask()` method |
| `ask_retry()` | Retry on failure | The `LLM.ask_retry()` method |
| The server-check code | List the models | The `check_server()` function |

**리팩터링**이란 동작은 그대로 두고 구조만 바꾸는 일입니다.
새 기능을 넣는 것이 아니라, 있던 것을 쓰기 좋게 정리하는 것입니다.

> **EN** — **Refactoring** means changing the structure while leaving the
> behaviour alone. You are not adding features; you are making what already
> exists pleasant to use.

---
## 2. 1단계 — 설정과 키 관리 · Step 1 — configuration and key handling

Colab의 `%%writefile` 매직은 셀 내용을 그대로 파일로 저장합니다.
`-a` 를 붙이면 이어붙이기입니다.

> `%%writefile` 은 **셀의 첫 줄**이어야 합니다.

> **EN** — Colab's `%%writefile` magic saves the cell's contents to a file as is;
> adding `-a` appends instead. `%%writefile` must be **the first line of the
> cell**.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

EN — Shared LLM wrapper for Computational Experiments 2: Building AI Agents.
"""

import os
import time

# ---- 서버 설정 / Server configuration --------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Secrets 의 KMU_BASE_URL 에서 읽는다.
# 이 파일은 공개 저장소에 올라간다.
# EN — The department URL is never written here; it is read from the secret
#      KMU_BASE_URL. This file gets pushed to a public repository.
PROVIDERS = {
    "kmu": {
        "base_url": None,                  # 실행 시점에 Secrets 에서 읽음 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model":    "gemma4",
        "key_name": "KMU_API_KEY",
        "label":    "계명대 학과 서버",
    },
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "base_url_key": None,
        "model":    "gemini-2.0-flash",
        "key_name": "GEMINI_API_KEY",
        "label":    "Google Gemini (백업)",
    },
}


# ---- 비밀정보 / Secrets ----------------------------------------------
def get_secret(name, secret=True):
    """Colab Secrets -> 환경변수 -> 직접 입력 순으로 찾는다.

    EN — Look in Colab Secrets, then the environment, then prompt.
    """
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    if os.environ.get(name):
        return os.environ[name].strip()

    if secret:
        import getpass
        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()
    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value
    return value


get_key = get_secret


def get_base_url(provider="kmu"):
    """서버 주소. 끝에 /chat/completions 가 붙어 있어도 잘라낸다.

    EN — The server URL, with a trailing /chat/completions trimmed off.
    """
    conf = PROVIDERS[provider]
    if conf["base_url"]:
        return conf["base_url"].rstrip("/")
    url = get_secret(conf["base_url_key"], secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    conf["base_url"] = url
    return url

In [ ]:
# 잘 만들어졌는지 확인 / Check that the file was written correctly
!head -20 kmu_llm.py
print("...")
!wc -l kmu_llm.py

---
## 3. 2단계 — `LLM` 클래스 · Step 2 — the `LLM` class

1주차에는 `client` 라는 전역 변수와 `ask` 함수가 따로 놀았습니다.
클래스로 묶으면 **설정과 동작이 한 덩어리**가 되어, 서버를 여러 개 동시에
쓰는 것도 자연스러워집니다.

> **EN** — In week 1 a global `client` and a free-standing `ask` function drifted
> apart. Binding them into a class puts **configuration and behaviour in one
> piece**, and running several servers at once becomes natural.

In [ ]:
%%writefile -a kmu_llm.py


# ---- LLM 클라이언트 / LLM client -------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.
    """

    def __init__(self, provider="kmu", model=None, base_url=None,
                 api_key=None, timeout=120.0):
        if provider not in PROVIDERS:
            raise ValueError(f"알 수 없는 provider: {provider}")
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label    = conf["label"]
        self.model    = model    or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key  = api_key  or get_key(conf["key_name"])

        from openai import OpenAI
        self.client = OpenAI(api_key=self.api_key,
                             base_url=self.base_url, timeout=timeout)

    def __repr__(self):
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    def raw(self, messages, **kw):
        """응답 객체 전체. usage(토큰 수) 확인용.

        EN — The whole response object, so you can read `usage`.
        """
        params = {"model": self.model, "messages": messages,
                  "temperature": 0.7, "max_tokens": 1024}
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages, **kw):
        """응답 텍스트만. / The response text only."""
        response = self.raw(messages, **kw)
        return (response.choices[0].message.content or "").strip()

    def ask(self, prompt, system=None, **kw):
        """한 번짜리 질문. / A one-shot question."""
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(self, prompt, system=None, n_retry=3, wait=2.0, **kw):
        """일시적 오류를 재시도한다. 반복 실험에서 필수.

        EN — Retry transient errors. Essential for repeated experiments.
        """
        last = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:
                last = exc
                if attempt < n_retry:
                    print(f"  [재시도 {attempt}] {type(exc).__name__}")
                    time.sleep(wait * attempt)
        raise RuntimeError(f"{n_retry}회 모두 실패") from last

    def stream(self, messages, **kw):
        """응답을 조각 단위로 흘려보낸다. / Yield the response chunk by chunk."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content

---
## 4. 3단계 — 편의 함수와 서버 진단 · Step 3 — convenience functions and the server check

`ask("질문")` 한 줄로 쓸 수 있게 모듈 수준 함수를 하나 둡니다.
객체를 매번 만들지 않도록 안에서 재사용합니다.

> **EN** — Add one module-level function so a single line `ask("…")` works. It
> reuses one object internally rather than building a new one each time.

In [ ]:
%%writefile -a kmu_llm.py


# ---- 편의 함수 / Convenience functions --------------------------------
_default_llm = None


def ask(prompt, system=None, provider="kmu", **kw):
    """가장 간단한 한 줄 호출. / The simplest one-line call."""
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# ---- 서버 진단 / Server diagnostics -----------------------------------
def check_server(provider="kmu", base_url=None):
    """서버가 살아 있는지, 어떤 모델이 있는지 확인한다.

    EN — Check that the server is alive and see which models it serves.
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    print("조회 중: GET <서버주소>/models")   # 주소는 화면에 남기지 않는다 / never show the URL
    try:
        r = requests.get(f"{base_url}/models",
                         headers={"Authorization": f"Bearer {api_key}"},
                         timeout=15)
    except Exception as exc:
        print(f"[실패] 연결 불가: {type(exc).__name__}: {exc}")
        print("  - Secrets 의 KMU_BASE_URL 이 정확한지, 끝이 /v1 인지 확인하세요.")
        print("  - 계속 안 되면 LLM(provider='gemini') 를 쓰세요.")
        return {"ok": False, "models": []}

    if r.status_code != 200:
        print(f"[실패] HTTP {r.status_code}: {r.text[:200]}")
        if r.status_code in (401, 403):
            print("  -> API 키를 확인하세요.")
        elif r.status_code == 404:
            print("  -> KMU_BASE_URL 끝의 /v1 을 확인하세요.")
        return {"ok": False, "models": []}

    models = [m["id"] for m in r.json().get("data", [])]
    print(f"[성공] 모델 {len(models)}개")
    for name in models:
        print("   -", name, "  <-- 기본" if name == conf["model"] else "")

    if conf["model"] not in models and models:
        print(f"\n[주의] '{conf['model']}' 이 목록에 없습니다. PROVIDERS 를 수정하세요.")

    return {"ok": True, "models": models}

In [ ]:
# 완성. 전체 구조 확인 / Done. Inspect the overall structure
!wc -l kmu_llm.py
!grep -n "^class \|^def \|    def " kmu_llm.py

---
## 5. 써 보기 · Using it


In [ ]:
!pip install -q openai

import kmu_llm

print(kmu_llm.check_server())

In [ ]:
from kmu_llm import LLM, ask

# 방법 1: 한 줄 / Option 1: one line
print(ask("표준오차를 한 문장으로 설명해 주세요.", max_tokens=200))
print()

# 방법 2: 객체를 만들어서 (설정을 바꿔 쓸 때)
# EN — Option 2: build an object, for when you want to vary the configuration
llm = LLM(provider="kmu")
print(llm)
print(llm.ask("자유도를 한 문장으로 설명해 주세요.", temperature=0.0, max_tokens=200))

> 지난주 노트북과 비교해 보세요.
> **`get_key`, `OpenAI(...)`, `ask` 정의가 전부 사라졌습니다.**
> 앞으로 새 노트북에서는 두 줄이면 됩니다.

> **EN** — Compare with last week's notebooks: **the definitions of `get_key`,
> `OpenAI(...)` and `ask` have all vanished.** From here on, a new notebook needs
> two lines.

---
## 6. 내 저장소에 올리기 · Pushing to your own repository


In [ ]:
import os, shutil, subprocess

def get_secret(name):
    # EN — Colab Secrets -> environment variable -> prompt.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    import getpass
    v = getpass.getpass(f"{name} 를 입력하세요: ").strip()
    os.environ[name] = v
    return v

GITHUB_USER  = get_secret("GITHUB_USER")
GITHUB_TOKEN = get_secret("GITHUB_TOKEN")
REPO_NAME    = "stat-agent-2026"      # <-- 자기 저장소 이름 / your own repository name

print("사용자:", GITHUB_USER, "| 저장소:", REPO_NAME)

In [ ]:
%cd /content
!rm -rf /content/{REPO_NAME}

url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
print(r.stderr.replace(GITHUB_TOKEN, "***"))

# 방금 만든 모듈을 src/ 에 복사 (폴더가 없으면 만든다)
# EN — Copy the module we just wrote into src/, creating the folder if needed
os.makedirs(f"/content/{REPO_NAME}/src", exist_ok=True)
shutil.copy("/content/kmu_llm.py", f"/content/{REPO_NAME}/src/kmu_llm.py")
%cd /content/{REPO_NAME}
!git status --short

In [ ]:
!git add src/kmu_llm.py
!git commit -m "kmu_llm 모듈 추가 - LLM 클래스, ask, check_server"

r = subprocess.run(["git", "push"], capture_output=True, text=True)
print(r.stderr.replace(GITHUB_TOKEN, "***"))
print("종료코드:", r.returncode)

---
## 7. 진짜 시험 — 런타임을 초기화하고 다시 해 보기 · The real test — reset the runtime and try again

지금 `import kmu_llm` 이 되는 것은 **파일이 이 런타임에 있기 때문**입니다.
이것은 증명이 아닙니다.

> "제 컴퓨터에서는 되는데요" 라는 말이 나오는 이유가 대부분 여기에 있습니다.

**지금 하세요:**

메뉴 **런타임 → 세션 다시 시작 및 모두 실행 취소** (또는 **런타임 초기화**)

`/content` 의 파일이 전부 사라집니다. 그 다음 아래 셀부터 실행하세요.

> **EN** — `import kmu_llm` works right now only **because the file happens to be
> in this runtime**. That is not proof. This is very often where "but it works on
> my machine" comes from.
>
> **Do it now:** menu **Runtime → Restart session and clear all outputs** (or
> **Reset runtime**). Everything in `/content` disappears. Then start from the
> cell below.

In [ ]:
# 런타임 초기화 후 이 셀부터 실행 / Start here after resetting the runtime
!ls /content
print("---")

# kmu_llm.py 가 사라졌는지 확인 / Check that kmu_llm.py is really gone
try:
    import kmu_llm
    print("아직 남아 있습니다. 런타임을 초기화하지 않으셨네요.")
except ModuleNotFoundError:
    print("깨끗합니다. 이제 GitHub에서 내려받겠습니다.")

In [ ]:
# 내 저장소에서 내려받기
# raw.githubusercontent.com 주소를 써야 파일 원문이 옵니다.
# (github.com 주소를 쓰면 HTML 페이지가 내려옵니다)
# EN — Download from your own repository. You must use the
#      raw.githubusercontent.com URL to get the file itself; a github.com URL
#      gives you an HTML page instead.

import os

def get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    return input(f"{name}: ").strip()

GITHUB_USER = get_secret("GITHUB_USER")
REPO_NAME   = "stat-agent-2026"      # <-- 자기 저장소 이름 / your own repository name

RAW = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/main/src/kmu_llm.py"
print(RAW)

!wget -q -O kmu_llm.py "{RAW}"
!head -3 kmu_llm.py

In [ ]:
!pip install -q openai

from kmu_llm import ask, check_server
check_server()
print()
print(ask("성공했습니다. 축하 인사를 한 문장으로 해 주세요.", max_tokens=100))

> **이제 어떤 Colab 노트북에서든 두 줄로 시작할 수 있습니다.**
>
> ```python
> !wget -q -O kmu_llm.py https://raw.githubusercontent.com/<계정>/<저장소>/main/src/kmu_llm.py
> from kmu_llm import ask
> ```

> **EN — you can now start any Colab notebook in two lines**, using the same
> `wget` and `import` shown above.

---
## 8. 함정 — 파일을 고쳤는데 반영이 안 된다 · The trap — you edited the file and nothing changed

파이썬은 한 번 `import` 한 모듈을 캐시에 담아 둡니다.
파일을 고쳐도 다시 `import` 하면 **옛날 것이 그대로 나옵니다.**

에이전트를 개발하면서 `kmu_llm.py` 를 수십 번 고치게 되므로
반드시 알아야 하는 함정입니다.

> **EN** — Python caches a module once it has been imported, so editing the file
> and importing again **still gives you the old one**. You will edit
> `kmu_llm.py` dozens of times while building your agent, so this trap is one you
> have to know.

In [ ]:
# 파일에 함수를 하나 추가 / Append one function to the file
with open("kmu_llm.py", "a", encoding="utf-8") as f:
    f.write("\n\ndef 인사():\n    return '안녕하세요'\n")

import kmu_llm
try:
    print(kmu_llm.인사())
except AttributeError as exc:
    print("AttributeError:", exc)
    print("-> 파일에는 있는데 모듈에는 없습니다. 캐시 때문입니다.")

In [ ]:
# 해결: reload / The fix: reload
import importlib
importlib.reload(kmu_llm)

print(kmu_llm.인사())

| 상황 | 해결 |
|---|---|
| 함수 내용만 바꿨다 | `importlib.reload(kmu_llm)` |
| `from kmu_llm import ask` 로 가져온 이름을 바꿨다 | reload 후 **다시 import** 해야 함 |
| 클래스를 바꿨는데 이상하다 | **런타임 재시작**이 확실 |

**EN**

| Situation | Fix |
|---|---|
| You only changed a function body | `importlib.reload(kmu_llm)` |
| You changed a name imported via `from kmu_llm import ask` | reload, then **import again** |
| You changed a class and things look strange | **restart the runtime** — the sure fix |

> 헷갈리면 그냥 런타임을 재시작하십시오. 시간을 아끼는 길입니다.
> **EN** — When in doubt, just restart the runtime. It is the faster route.

---
## 9. 수업 저장소의 완성본과 비교하기 · Comparing with the course reference version

이 수업의 완성본도 준비되어 있습니다.
여러분이 만든 것과 비교해 보세요. 3주차부터는 이 파일을 확장해 나갑니다.

아래 준비 셀이 완성본을 `kmu_llm_official.py` 로 만들어 줍니다.
여러분이 만든 `kmu_llm.py` 와 나란히 놓고 비교해 보세요.

The setup cell below writes the reference version as `kmu_llm_official.py`
so you can compare it with your own. From week 3 on we keep extending this file.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm_official.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

EN — Shared LLM wrapper for "Computational Experiments 2: Building AI Agents".
Calls the Keimyung Statistics department server and Google Gemini the same way.
Both expose an OpenAI-compatible API, so we use the single openai SDK and only
swap `base_url` and `model`.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기

 EN — This repository is PUBLIC, so never write the department server URL or
 the API key in code. Both are announced in class; store them in Colab Secrets.
     Key icon (🔑) on the left → new secret → turn on notebook access
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예 · Example:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
                                         # Check the server is alive first
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로 / Fallback route
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정 / Server configuration
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
# EN — The department server URL is never written here; it is read from the
#      Colab secret KMU_BASE_URL. The model tag is harmless to publish, so it
#      stays a constant.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
# EN — Fallback route. Get a free key from Google AI Studio. (public URL)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기 / Loading secrets
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    EN — Look up a configuration value in a safe order:
    1) Colab Secrets (key icon on the left)  2) environment variable  3) prompt.
    The point is to keep the value out of the source. The moment a notebook or
    module is pushed to GitHub, anything written in the code is world-readable —
    the internal server URL just as much as the API key.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
             EN — True hides the typed value (getpass). Pass False for values
             like a URL where you need to see typos.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수 / environment variable
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력 / type it in
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다 / ask only once per runtime
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)

    EN — Fetch the API key. Alias of get_secret, kept under the week-1 name.
    """
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.

    EN — Return the server URL with any trailing '/' removed. The department
    URL is never stored in the repository, so it comes from Secrets. If you
    saved the full '/chat/completions' path, it is trimmed for you.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    # EN — The openai SDK appends /chat/completions itself, so strip it here in
    #      case the whole path was saved as the secret.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용 / reuse within the session
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트 / LLM client
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
        EN — Which server to use. Defaults to the department server.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
        EN — Override the defaults. You normally leave these alone.
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        # EN — Never print the URL; screen shares and screenshots leak it easily.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 / Core calls -------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용.

        EN — Return the full response object, so you can inspect `usage`
        (token counts).
        """
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다.

        EN — Send the message list and return only the response text.
        """
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과.

        EN — Yield the response chunk by chunk, giving the typing effect.
        """
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 / Convenience methods ---------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다.

        EN — A one-shot question, for when building `messages` by hand is
        tedious.
        """
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.

        EN — Retry transient network/server errors, so that a lab with many
        calls (a 30-repetition experiment, say) does not die on one failure.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반 / any network or server error
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다 / back off progressively
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다.

        EN — Ask the server how many input tokens the messages use; the output
        is capped at 1 token to avoid waste.
        """
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수 / Module-level convenience functions
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.

    EN — The simplest one-line call. Internally it builds one LLM object and
    reuses it.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단 / Server diagnostics
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    EN — Check that the server is alive and see which models it serves. This is
    the very first lab of the course; on failure it prints what to check.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    # EN — Never print the full URL; show the host masked.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다.

    EN — Mask the host so screen shares and screenshots do not leak it.
    """
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들. / What to check when the connection fails."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
!echo "=== 줄 수 비교 / line-count comparison ==="
!wc -l kmu_llm.py kmu_llm_official.py
!echo
!echo "=== 공식판에만 있는 것 / what only the reference version has ==="
!grep -n "^def \|^class \|    def " kmu_llm_official.py

### 공식판이 더 하고 있는 것 · What the reference version adds

| 추가된 것 | 왜 |
|---|---|
| 타입 힌트 (`-> str`) | 편집기가 오류를 미리 잡아 준다 |
| `__all__` | `from kmu_llm import *` 시 공개할 이름을 명시 |
| `prompt=False` 옵션 | 자동 실행 환경에서 입력 대기로 멈추지 않게 |
| 상세한 오류 안내 (`_print_troubleshooting`) | 수업 중 막혔을 때 스스로 해결하도록 |
| `KMU_BASE_URL` 을 환경변수로 덮어쓰기 | 서버 주소가 바뀌어도 코드 수정 불필요 |

**EN**

| What was added | Why |
|---|---|
| Type hints (`-> str`) | The editor catches mistakes before you run |
| `__all__` | States which names `from kmu_llm import *` exposes |
| A `prompt=False` option | So an automated run never blocks waiting for input |
| Detailed error guidance (`_print_troubleshooting`) | So you can unstick yourself during class |
| `KMU_BASE_URL` overridable by environment variable | No code change when the server URL moves |

지금 다 이해하지 않아도 됩니다. 필요해질 때 다시 보십시오.

> **EN** — You do not have to understand all of it now. Come back when you need it.

---
## 연습문제 · Exercises

**문제 1.** `kmu_llm.py` 에 `ask_json()` 의 **빈 껍데기**를 추가하고 커밋하시오.
3주차에 내용을 채웁니다.
```python
def ask_json(prompt, schema=None, n_retry=3, **kw):
    """모델에게 JSON을 받아 파싱해서 dict로 돌려준다. (3주차에 구현)"""
    raise NotImplementedError
```

**문제 2.** `LLM` 클래스에 `count_tokens(messages)` 메서드를 추가하시오.
`max_tokens=1` 로 호출해 `usage.prompt_tokens` 를 반환하면 됩니다.

**문제 3.** `PROVIDERS` 에 세 번째 항목을 추가하되,
학과 서버의 **다른 모델**(`check_server()` 목록에 있는 것)을 가리키게 하시오.

**문제 4.** 지금은 `wget` 으로 파일 하나만 받습니다.
파일이 여러 개로 늘어나면 어떻게 해야 하는가? `git clone` 과 비교해 장단점을 적으시오.

> **EN — 1.** Add an **empty shell** of `ask_json()` to `kmu_llm.py` and commit
> it; we fill it in during week 3.
>
> **EN — 2.** Add a `count_tokens(messages)` method to the `LLM` class: call with
> `max_tokens=1` and return `usage.prompt_tokens`.
>
> **EN — 3.** Add a third entry to `PROVIDERS` pointing at **a different model**
> on the department server (one from the `check_server()` list).
>
> **EN — 4.** Right now `wget` fetches a single file. What do you do when there
> are several? Write down the pros and cons against `git clone`.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 2.** 예시
```python
    def count_tokens(self, messages):
        r = self.raw(messages, max_tokens=1)
        return r.usage.prompt_tokens
```
입력 토큰만 알면 되므로 출력은 1토큰으로 잘라 낭비를 줄입니다.
정확히 하려면 `tiktoken` 같은 로컬 토크나이저를 써야 하지만,
모델마다 토크나이저가 달라 서버에 물어보는 쪽이 확실합니다. (4주차)

**문제 4.**

| | `wget` 파일 하나 | `git clone` |
|---|---|---|
| 속도 | 빠름 | 느림 |
| 파일 여러 개 | 매번 wget 해야 함 | 한 번에 |
| 버전 지정 | 어려움 | 태그/커밋으로 가능 |
| 수정 후 push | 불가 | 가능 |

**실습 노트북에서 읽기만 할 때는 `wget`, 코드를 고쳐서 올릴 때는 `clone`** 이
적당합니다. 프로젝트가 커지면 `pip install git+https://...` 로 패키지처럼
설치하는 방법도 있습니다.

---

**EN — 2.** We only need the input count, so capping the output at one token
avoids waste. To be exact you would use a local tokenizer such as `tiktoken`, but
tokenizers differ per model, so asking the server is the reliable route (week 4).

**EN — 4.**

| | `wget`, one file | `git clone` |
|---|---|---|
| Speed | Fast | Slow |
| Several files | A `wget` for each | All at once |
| Pinning a version | Awkward | By tag or commit |
| Editing and pushing back | Not possible | Possible |

**Use `wget` when a lab notebook only reads the code, and `clone` when you will
edit and push it back.** As a project grows, installing it like a package with
`pip install git+https://...` becomes another option.

</details>

---
## 정리 · Summary

- 반복되는 코드는 **파일 하나**로 모은다 (리팩터링)
- 자기 저장소에 올리고, **런타임을 초기화해서** 다시 되는지 확인한다
- `raw.githubusercontent.com` 주소로 받아야 파일 원문이 온다
- 모듈을 고쳤으면 `importlib.reload` 또는 **런타임 재시작**

> **EN**
> - Gather repeated code into **a single file** (refactoring)
> - Push it to your own repository, then **reset the runtime** and confirm it
>   still works
> - Fetch from the `raw.githubusercontent.com` URL to get the file itself
> - After editing a module, `importlib.reload` — or **restart the runtime**

**다음 노트북** → [`06_appendix_secrets_and_config.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week02/06_appendix_secrets_and_config.ipynb)